# Collecting EODHD Stock Performance Data

## Purpose
Collect daily OHLCV (Open, High, Low, Close, Volume) and return data for all G-SIB bank tickers over the historical period (2014-02-15 onwards). This data will be used for regression analysis against news topics (following Chen 2025 methodology).

## Inputs
- EODHD API credentials (via `.env`)
- Bank ticker list: 30 Global Systemically Important Banks (G-SIBs)

## Outputs
- `Data/bank_tickers_performance.csv` - All bank ticker returns
- `Data/sp500_performance.csv` - S&P 500 benchmark for comparison

## Notes
- Daily returns calculated as percentage change in closing price
- Data aligned by trading date for downstream regression
- S&P 500 included as market benchmark/control variable


In [ ]:
import os
import pandas as pd
import requests
from dotenv import load_dotenv
from datetime import datetime, timedelta
import time

load_dotenv()

API_KEY = os.getenv('EODHD_API_KEY')

bank_tickers = ['JPM', 'BAC', 'C', 'HSBC', 'IDCBY', 'GS', 'BNPQY', 'UBS', 'ACGBY', 'BACHY', 'CICHY', 'BCS', 'MUFG', 'MS', 'WFC', 'BK', 'STT', 'DB', 'ING', 'SAN', 'RY', 'TD', 'SCBFF', 'SCGLY', 'MFG', 'SMFG', 'BCMXY', 'GCRLY', 'BPCE']

# Configuration for historical data collection
START_DATE = '2014-02-15'
END_DATE = datetime.now().strftime('%Y-%m-%d')

# Collect stock performance data for all bank tickers
all_returns = []

for ticker in bank_tickers:
    print(f"Fetching performance data for {ticker}...")
    
    # EODHD EOD data endpoint
    url = f"https://eodhd.com/api/eod/{ticker}"
    params = {
        'from': START_DATE,
        'to': END_DATE,
        'api_token': API_KEY,
        'fmt': 'json'
    }
    
    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            
            if isinstance(data, list) and len(data) > 0:
                print(f"  > Found {len(data)} trading days")
                
                # Add ticker to each record
                for record in data:
                    record['ticker'] = ticker
                    # Calculate daily return if close price exists
                    if 'close' in record:
                        record['close_price'] = record['close']
                
                all_returns.extend(data)
            else:
                print(f"  > No data found for {ticker}")
                
        elif response.status_code == 429:
            print("Rate limit hit! Sleeping for 60 seconds...")
            time.sleep(60)
            continue
        else:
            print(f"  > Error {response.status_code}: {response.text}")
    
    except Exception as e:
        print(f"  > Connection error: {e}")
        time.sleep(5)
    
    # Rate limit: small delay between requests
    time.sleep(0.5)

# --- SAVE OUTPUT ---
if all_returns:
    df = pd.DataFrame(all_returns)
    
    # Keep relevant columns for regression analysis
    # OHLC data + volume + sentiment (if available) + ticker identifier
    cols_to_keep = ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'adjusted_close']
    
    # Only keep columns that exist in the data
    cols_to_keep = [col for col in cols_to_keep if col in df.columns]
    
    df = df[cols_to_keep]
    
    # Convert date to datetime
    df['date'] = pd.to_datetime(df['date'])
    
    # Sort by date and ticker
    df = df.sort_values(['date', 'ticker'])
    
    # Calculate daily returns (percentage change in close price)
    df['daily_return'] = df.groupby('ticker')['close'].pct_change() * 100
    df['daily_adj_return'] = df.groupby('ticker')['adjusted_close'].pct_change() * 100 if 'adjusted_close' in df.columns else None
    
    # Save to CSV
    df.to_csv('Data/eodhd_tickers_performance.csv', index=False)
    print(f"\nSUCCESS: Saved {len(df)} records to Data/eodhd_tickers_performance.csv")
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
    print(f"\nTickers covered: {df['ticker'].nunique()} banks")
    print(f"\nFirst few rows:")
    print(df.head())
else:
    print("No data was retrieved.")


Fetching performance data for JPM...
  > Found 3075 trading days
  > Found 3075 trading days
Fetching performance data for BAC...
Fetching performance data for BAC...
  > Found 3075 trading days
  > Found 3075 trading days
Fetching performance data for C...
Fetching performance data for C...
  > Found 3075 trading days
  > Found 3075 trading days
Fetching performance data for HSBC...
Fetching performance data for HSBC...
  > Found 3075 trading days
  > Found 3075 trading days
Fetching performance data for IDCBY...
Fetching performance data for IDCBY...
  > Found 3071 trading days
  > Found 3071 trading days
Fetching performance data for GS...
Fetching performance data for GS...
  > Found 3075 trading days
  > Found 3075 trading days
Fetching performance data for BNPQY...
Fetching performance data for BNPQY...
  > Found 3074 trading days
  > Found 3074 trading days
Fetching performance data for UBS...
Fetching performance data for UBS...
  > Found 3075 trading days
  > Found 3075 tradin

In [3]:

# Data Validation and Summary Statistics

print("\n" + "="*60)
print("Data Validation & Summary Statistics")
print("="*60)

# Load the saved bank performance data
bank_perf = pd.read_csv('Data/eodhd_tickers_performance.csv')
bank_perf['date'] = pd.to_datetime(bank_perf['date'])

print("\n--- BANK TICKERS PERFORMANCE ---")
print(f"Total records: {len(bank_perf)}")
print(f"Date range: {bank_perf['date'].min()} to {bank_perf['date'].max()}")
print(f"Number of banks: {bank_perf['ticker'].nunique()}")
print(f"Banks: {sorted(bank_perf['ticker'].unique())}")

print("\n--- RECORDS PER BANK ---")
records_per_bank = bank_perf['ticker'].value_counts().sort_index()
print(records_per_bank)

print("\n--- MISSING DATA ---")
print(bank_perf.isnull().sum())

print("\n--- RETURN STATISTICS ---")
print(bank_perf.groupby('ticker')['daily_return'].describe().round(4))

print("\n--- OVERALL CORRELATION SUMMARY ---")
# Check correlation between returns and market
if 'daily_return' in bank_perf.columns:
    print(f"Mean daily return across all banks: {bank_perf['daily_return'].mean():.4f}%")
    print(f"Std dev of daily returns: {bank_perf['daily_return'].std():.4f}%")
    print(f"Min daily return: {bank_perf['daily_return'].min():.4f}%")
    print(f"Max daily return: {bank_perf['daily_return'].max():.4f}%")

print("\n✅ Data collection complete. Ready for regression analysis against news topics.")



Data Validation & Summary Statistics

--- BANK TICKERS PERFORMANCE ---
Total records: 80295
Date range: 2014-02-18 00:00:00 to 2026-05-08 00:00:00
Number of banks: 27
Banks: ['ACGBY', 'BAC', 'BACHY', 'BCMXY', 'BCS', 'BK', 'BNPQY', 'C', 'CICHY', 'DB', 'GS', 'HSBC', 'IDCBY', 'ING', 'JPM', 'MFG', 'MS', 'MUFG', 'RY', 'SAN', 'SCBFF', 'SCGLY', 'SMFG', 'STT', 'TD', 'UBS', 'WFC']

--- RECORDS PER BANK ---
ticker
ACGBY    3070
BAC      3075
BACHY    3074
BCMXY     951
BCS      3075
BK       3075
BNPQY    3074
C        3075
CICHY    3070
DB       3075
GS       3075
HSBC     3075
IDCBY    3071
ING      3075
JPM      3075
MFG      3075
MS       3075
MUFG     3075
RY       3075
SAN      3075
SCBFF    2489
SCGLY    3071
SMFG     3075
STT      3075
TD       3075
UBS      3075
WFC      3075
Name: count, dtype: int64

--- MISSING DATA ---
date               0
ticker             0
open               0
high               0
low                0
close              0
volume             0
adjusted_close    